# VTS O&M Manual — book builder (v3)
Lee `data/VTS_manual_content.xlsx` (Excel o Google Sheets) y genera:
- el manual en PDF,
- los checklists de inspección en Excel,
- los diagramas de flujo de las tareas (a partir del texto de la hoja `Flow_specs`).

**Cómo se usa:** `Run all`. Los textos, el orden y los diagramas se editan en el Excel, no aquí.
Imágenes: `images/<Image ID>.png|jpg|svg`. Sin archivo se dibuja un cuadrado marcador.
Para los diagramas hace falta Graphviz (en Colab ya viene; en Mac: `brew install graphviz`).

In [ ]:
# Solo la primera vez: descomenta
# !pip install weasyprint markdown openpyxl
# !apt-get install -y graphviz

In [ ]:
# --- Configuración ---------------------------------------------------------
import os, re, json, html, shutil, pathlib, sys
BASE_DIR   = pathlib.Path(os.environ.get("VTS_DIR", "."))
EXCEL_PATH = BASE_DIR / "data" / "VTS_manual_content.xlsx"
IMAGES_DIR = BASE_DIR / "images"
OUT_DIR    = BASE_DIR / "output"
FLOWS_DIR  = OUT_DIR / "flows"
OUT_PDF    = OUT_DIR / "VTS_D01_O&M_Manual_v5.pdf"
PAGE_SIZE  = "Letter"          # "Letter" o "A4"
DRAFT_MODE = True              # True: imprime todo y marca lo no validado. False: solo tareas con Engineering validation = Validated
VERSION    = "Version 1.0 (reorganized draft v5)"
BLUE, BLUE_LIGHT, YELLOW, INK = "#03467E", "#E6F0F9", "#F6B800", "#1F2D3D"
SYS_NAME  = {"SB":"Sediment Basin","TW":"Treatment Wetland","VD":"Vegetated Ditch","WB":"Woodchip Bioreactor","COR":"Corrective Maintenance"}
SYS_COLOR = {"SB":"#2B7BBF","TW":"#3C9D5B","VD":"#C99700","WB":"#DD6B2A","COR":"#B4361F"}
for d in (OUT_DIR, FLOWS_DIR, IMAGES_DIR): d.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(BASE_DIR))

In [ ]:
# --- Datos (Excel) -----------------------------------------------------------------------------
from openpyxl import load_workbook
def rows(wb, name):
    ws = wb[name]; it = ws.iter_rows(values_only=True); head = [h for h in next(it)]
    return [dict(zip(head, r)) for r in it if any(c not in (None, "") for c in r)]
S = lambda v: "" if v is None else str(v).strip()
def split(v, sep=";"): return [x.strip() for x in S(v).split(sep) if x.strip()]
def load_excel(path):
    wb = load_workbook(path, data_only=True)
    steps_raw = rows(wb, "Steps"); flows = {r["Task ID"]: r for r in rows(wb, "Flow_specs")}
    imgs = [dict(id=S(r["Image ID"]), caption=S(r["Caption"]), label=S(r["Label (v1.0)"]), page=r["Page (v1.0)"], type=S(r["Type"]), task=S(r["Task ID"]), file=S(r["File"])) for r in rows(wb, "Images")]
    tasks = []
    for r in rows(wb, "Tasks"):
        i = S(r["Task ID"]); fl = flows.get(i, {})
        tasks.append(dict(id=i, system=S(r["System"]), number=int(r["No"]), name=S(r["Task name"]), kind=S(r["Kind"]), primary=S(r["Primary frequency"]), freq=split(r["Frequency chips"]),
            type=S(r["Task type"]), level=S(r["Action level"]), short=S(r["Short description"]), fnotes=S(r["When"]), resp=S(r["Responsibilities"]), tools=S(r["Tools and equipment"]),
            safety=S(r["Safety notes"]), waste=S(r["Waste handling"]), esc=S(r["Escalation"]), notes=S(r["Notes"]), eng=S(r["Engineering validation"]) or "Pending",
            images=[x["id"] for x in imgs if x["task"] == i], flow_status=S(fl.get("Status")), flow_spec=S(fl.get("Spec")),
            visit=S(r.get("Visit")), corrective=S(r.get("Corrective class")), component=S(r.get("Component")), f02=S(r.get("F02 summary")), months=S(r.get("Plan months")), basis=[x.strip() for x in S(r.get("Basis")).split(";") if x.strip()]))
    steps = [dict(task=S(r["Task ID"]), order=int(r["Order"]), level=int(r["Level"]), type=S(r["Step type"]), text=S(r["Text"])) for r in steps_raw]
    topics = [dict(id=S(r["Topic ID"]), name=S(r["Name"]), title=S(r["Title"]), system=S(r["System"]), intro=S(r["Intro"]), newpage=S(r["Starts new page"]) == "Y") for r in rows(wb, "Topics")]
    sections = [dict(key=S(r["Key"]), topic=S(r["Topic ID"]), order=int(r["Order"]), type=S(r["Type"]), heading=S(r["Heading"]), text=S(r["Text"]), table=S(r["Table data"]), images=split(r["Images"], ","), layout=S(r["Image layout"]), source=S(r["Table source"])) for r in rows(wb, "Topic_Sections")]
    cl = [dict(id=S(r["Item ID"]), system=S(r["System"]), section=S(r["Section"]), order=int(r["Order"]), question=S(r["Question"]), look=S(r["Look for"]), normal=S(r["Normal looks like"]),
               abnormal=S(r["If abnormal"]), visits=split(r["Visit types"]), photo=S(r["Photo if abnormal"]) == "Yes", go=split(r["Go to tasks"], "|"), notes=S(r["Notes"])) for r in rows(wb, "Checklist_Items")]
    look = {}
    for r in sorted(rows(wb, "Look_at"), key=lambda r: (S(r["System"]), int(r["Order"]))):
        look.setdefault(S(r["System"]), []).append(dict(section=S(r["Section"]), check=S(r["What to check"]), abnormal=S(r["If abnormal"]), go=[x for x in split(r["Go to tasks"], "|") if x != "—"]))
    return dict(tasks=tasks, steps=steps, topics=topics, sections=sections, images=imgs, checklist_items=cl, look_at=look,
        references=[dict(key=S(r["Citation key"]), apa=S(r["APA"]), short=S(r.get("Short cite"))) for r in rows(wb, "References")],
        site_params={k: [S(r["Field"]) for r in rows(wb, "Site_Params") if S(r["System"]) == k] for k in {S(r["System"]) for r in rows(wb, "Site_Params")}},
        glossary=[dict(term=S(r["Term"]), definition=S(r["Definition"])) for r in rows(wb, "Glossary")],
        manual_org=[(S(r["Item type"]), S(r["Ref"])) for r in sorted(rows(wb, "Manual_Order"), key=lambda r: r["Order"])],
        separators={S(r["Name"]): dict(title=S(r["Title"]), subtitle=S(r["Subtitle"]), body=S(r["Body"]), num=S(r["Num"]), version=S(r["Version"])) for r in rows(wb, "Separators")},
        documents={S(r["Code"]): S(r["Name"]) for r in rows(wb, "Documents")},
        doc_rows=[dict(code=S(r["Code"]), name=S(r["Name"]), type=S(r.get("Type")), desc=S(r.get("Description")), version=S(r.get("Version")), status=S(r.get("Status")), link=S(r.get("Link"))) for r in rows(wb, "Documents")],
        forms=[(S(r["Form"]), int(r["Order"]), S(r["Column"]), float(r["Width (cm)"])) for r in rows(wb, "Forms")])
D = load_excel(EXCEL_PATH)
TASK  = {t["id"]: t for t in D["tasks"]}; TOPIC = {t["id"]: t for t in D["topics"]}; IMG = {i["id"]: i for i in D["images"]}; DOC = D["documents"]
STEPS = {}
for s in sorted(D["steps"], key=lambda s: (s["task"], s["order"])): STEPS.setdefault(s["task"], []).append(s)
SECS = {}
for s in sorted(D["sections"], key=lambda s: s["order"]): SECS.setdefault(s["topic"], []).append(s)
CHECK = {}
for i in sorted(D["checklist_items"], key=lambda i: i["order"]): CHECK.setdefault(i["system"], []).append(i)
print({k: len(v) for k, v in dict(tareas=TASK, temas=TOPIC, pasos=D["steps"], secciones=D["sections"], imágenes=IMG, referencias=D["references"], checklist=D["checklist_items"], orden=D["manual_org"]).items()})

In [ ]:
# --- Diagramas de flujo (a partir de la hoja Flow_specs) -------------------------------------------------------
import flows
FLOW_SVG = {}
if shutil.which("dot") is None:
    print("[AVISO] Graphviz no está instalado: las tareas llevarán un cuadrado en lugar del diagrama.")
else:
    for t in D["tasks"]:
        if t["flow_status"] == "Curated" and t["flow_spec"]:
            try:
                flows.render_flow(t["flow_spec"], str(FLOWS_DIR / t["id"]), formats=("svg",)); FLOW_SVG[t["id"]] = FLOWS_DIR / f'{t["id"]}.svg'
            except Exception as e: print(f'[ERROR] diagrama de {t["id"]}: {e}')
print(f"{len(FLOW_SVG)} diagramas dibujados; {len(D['tasks']) - len(FLOW_SVG)} tareas con cuadrado marcador")

In [ ]:
# --- Revisiones de calidad (QA gates) ---------------------------------------------------------------------------
def qa_report():
    out = []; ids = [t["id"] for t in D["tasks"]]
    for i in set(ids):
        if ids.count(i) > 1: out.append(("ERROR", f"Task ID duplicado: {i}"))
    for t in D["tasks"]:
        if t["id"] != f'{t["system"]}_T{t["number"]:02d}': out.append(("ERROR", f'ID no coincide con sistema+número: {t["id"]}'))
        if not STEPS.get(t["id"]): out.append(("ERROR", f'{t["id"]} no tiene pasos'))
        for k in ("short", "resp", "tools", "safety", "esc"):
            if not t.get(k): out.append(("AVISO", f'{t["id"]}: campo vacío «{k}»'))
        if t["eng"] != "Validated": out.append(("INFO", f'{t["id"]}: validación de ingeniería = {t["eng"]}'))
    for t, r in D["manual_org"]:
        if t == "Task" and r not in TASK: out.append(("ERROR", f"Manual_Order apunta a tarea inexistente {r}"))
        if t == "Topic" and r not in TOPIC: out.append(("ERROR", f"Manual_Order apunta a tema inexistente {r}"))
    for tok in re.findall(r"\{\{(task|topic|doc):([^}]+)\}\}", json.dumps([D["tasks"], D["steps"], D["sections"], D["topics"]])):
        pool = {"task": TASK, "topic": TOPIC, "doc": DOC}[tok[0]]
        if tok[1] not in pool: out.append(("ERROR", f"Referencia rota {{{{{tok[0]}:{tok[1]}}}}}"))
    rk = {r["key"] for r in D["references"]}
    for t in D["tasks"]:
        for b in t["basis"]:
            if b not in rk: out.append(("ERROR", f'{t["id"]}: Basis apunta a una referencia inexistente «{b}»'))
        if t["flow_status"] == "Curated":
            try: flows.parse_flow(t["flow_spec"])
            except Exception as e: out.append(("ERROR", f'{t["id"]}: el texto del diagrama no se puede leer ({e})'))
    for it in D["checklist_items"]:
        for g in it["go"]:
            if g not in TASK: out.append(("ERROR", f'{it["id"]}: «Go to tasks» apunta a una tarea inexistente {g}'))
    for s_, rr in D["look_at"].items():
        for r_ in rr:
            for g in r_["go"]:
                if g not in TASK: out.append(("ERROR", f'Look_at {s_}: tarea inexistente {g}'))
    used = [x for t in D["tasks"] for x in t["images"]] + [x for s in D["sections"] for x in s["images"]]
    miss = [i for i in used if not any((IMAGES_DIR / f"{i}{e}").exists() for e in (".png", ".jpg", ".jpeg", ".svg"))]
    out.append(("AVISO", f"{len(miss)} de {len(used)} imágenes sin archivo (cuadrado marcador)"))
    for s in SYS_NAME:
        if s != "COR" and not CHECK.get(s): out.append(("AVISO", f"Sin ítems de checklist para {s}"))
    return out
ISSUES = qa_report()
for lvl in ("ERROR", "AVISO"):
    for l, m in ISSUES:
        if l == lvl: print(f"[{l}] {m}")
print(f'[INFO] {sum(1 for l, _ in ISSUES if l == "INFO")} tareas sin validar (se marcan como borrador)')
assert not any(l == "ERROR" for l, _ in ISSUES), "Hay errores: corrígelos en el Excel antes de generar el libro"

In [ ]:
# --- Utilidades de texto ----------------------------------------------------------------------------------------
import markdown, forms_xlsx
esc = html.escape
def xref(text):
    def rep(m):
        kind, key = m.group(1), m.group(2)
        if kind == "task":  return f'<a class="xref" href="#{key}">{esc(TASK[key]["name"])}</a>'
        if kind == "topic": return f'<a class="xref" href="#{key}">{esc(TOPIC[key]["title"])}</a>'
        return f'<a class="xref" href="#{key}">{esc(key)}</a>'
    return re.sub(r"\{\{(task|topic|doc):([^}]+)\}\}", rep, text)
def md(text, inline=False):
    t = xref(esc(text or "")); t = re.sub(r"(?m)([^\n])\n(- )", r"\1\n\n\2", t)
    out = markdown.markdown(t, extensions=["tables"])
    return re.sub(r"^<p>|</p>$", "", out) if inline else out
def field_html(text):
    if not text: return ""
    parts = re.split(r"\s*\*", text.strip()); head, notes = parts[0].strip(), [p.strip() for p in parts[1:] if p.strip()]
    h = md(head) if head else ""
    if notes: h += '<ul class="notes">' + "".join(f"<li>{md(n, True)}</li>" for n in notes) + "</ul>"
    return h
FIG = {"n": 0}; TAB = {"n": 0}
def placeholder(img_id, w="2.3in", h="2.3in", label="IMAGE"):
    for e in (".png", ".jpg", ".jpeg", ".svg"):
        p = IMAGES_DIR / f"{img_id}{e}"
        if p.exists(): return f'<img class="fig-img" src="{p.resolve().as_uri()}" style="max-width:100%;max-height:{h}">'
    return f'<div class="ph" style="width:{w};height:{h}"><span>{esc(label)}<br><b>{esc(img_id or "—")}</b><br>needs file</span></div>'
def figure(img_id, caption, w="2.3in", h="2.3in", cls=""):
    FIG["n"] += 1; cap = caption or IMG.get(img_id, {}).get("caption", "")
    return f'<figure class="{cls}">{placeholder(img_id, w, h)}<figcaption><b>Figure {FIG["n"]}.</b> {esc(cap)}</figcaption></figure>'
def flow_figure(tid, caption, max_h="6.5in", width=None):
    FIG["n"] += 1; w = f"width:{width};" if width else ""
    return f'<figure><img class="flowimg" src="{FLOW_SVG[tid].resolve().as_uri()}" style="{w}max-height:{max_h}"><figcaption><b>Figure {FIG["n"]}.</b> {esc(caption)}</figcaption></figure>'

In [ ]:
# --- Estilos (CSS) -----------------------------------------------------------------------------------------------------
FONT_DIR = (BASE_DIR / "fonts").resolve().as_uri()
CSS = f"""
@font-face {{ font-family:'Public Sans'; src:url('{FONT_DIR}/PublicSans-VF.ttf'); font-weight:100 900; font-style:normal; }}
@font-face {{ font-family:'Public Sans'; src:url('{FONT_DIR}/PublicSans-Italic-VF.ttf'); font-weight:100 900; font-style:italic; }}
@font-face {{ font-family:'Barlow Condensed'; src:url('{FONT_DIR}/BarlowCondensed-SemiBold.ttf'); font-weight:600; }}
@font-face {{ font-family:'Barlow Condensed'; src:url('{FONT_DIR}/BarlowCondensed-Bold.ttf'); font-weight:700; }}
:root {{ --blue:{BLUE}; --blue-l:{BLUE_LIGHT}; --yellow:{YELLOW}; --ink:{INK}; }}
@page {{ size:{PAGE_SIZE}; margin:0.7in 0.65in 0.8in 0.65in;
  @bottom-right {{ content:counter(page); font:700 10pt 'Barlow Condensed'; color:var(--blue); }}
  @bottom-left  {{ content:'Operation and Maintenance Manual · Vegetated Treatment Systems'; font:7.5pt 'Public Sans'; color:#6b7c8f; }} }}
@page cover {{ margin:0; @bottom-right {{content:none}} @bottom-left {{content:none}} }}
@page divider {{ margin:0; background:var(--blue); @bottom-right {{content:none}} @bottom-left {{content:none}} }}
@page credits {{ @bottom-right {{content:none}} @bottom-left {{content:none}} }}
@page landscape {{ size:{PAGE_SIZE} landscape; margin:0.42in 0.55in 0.6in; }}
html {{ font:9.2pt/1.42 'Public Sans', sans-serif; color:var(--ink); }}
h1,h2,h3 {{ font-family:'Barlow Condensed'; font-weight:700; color:var(--blue); margin:0; }}
h1 {{ font-size:23pt; text-transform:uppercase; letter-spacing:.2pt; line-height:1.05; border-bottom:2.2pt solid var(--blue); padding-bottom:3pt; margin-bottom:9pt; bookmark-level:1; bookmark-label:content(); page-break-after:avoid; }}
.eyebrow {{ font:600 9.5pt 'Barlow Condensed'; letter-spacing:1.2pt; text-transform:uppercase; color:#fff; display:inline-block; padding:1.5pt 7pt; border-radius:8pt; margin-bottom:5pt; }}
h2.sec {{ font-size:14pt; margin:11pt 0 4pt; page-break-after:avoid; }}
h2.sec.group {{ background:var(--yellow); color:#3a2a00; display:inline-block; padding:1pt 9pt; border-radius:3pt; font-size:12.5pt; }}
h2.sec.cap {{ font-size:12pt; border-left:4pt solid var(--yellow); padding-left:6pt; }}
p {{ margin:0 0 5.5pt; }} p.lead {{ page-break-after:avoid; }}
.intro {{ background:var(--blue-l); border-radius:6pt; padding:4pt 9pt; margin-bottom:6pt; }}
table {{ border-collapse:collapse; width:100%; margin:4pt 0 8pt; font-size:8.6pt; page-break-inside:auto; }}
th {{ background:var(--blue); color:#fff; text-align:left; padding:3.5pt 6pt; font-weight:700; }}
td {{ padding:3.5pt 6pt; border-bottom:.6pt solid #c9d6e3; vertical-align:top; }}
tr {{ page-break-inside:avoid; }} tbody tr:nth-child(even) td {{ background:#f5f9fc; }} td:first-child {{ font-weight:700; color:var(--blue); width:24%; }}
.keep {{ break-inside:avoid; }} thead {{ display:table-header-group; }}
tbody tr:last-child {{ page-break-before:avoid; }} tbody tr:first-child {{ page-break-after:avoid; }}
.src {{ font-size:7.8pt; color:#566a7f; }} .fn {{ font-size:7.8pt; color:#566a7f; border-top:.6pt solid #b9c8d8; padding-top:3pt; margin-top:8pt; }}
figure {{ margin:6pt 0; text-align:center; page-break-inside:avoid; }} figcaption {{ font-size:7.8pt; color:#566a7f; margin-top:3pt; }}
figure.float {{ float:left; width:2.75in; margin:2pt 14pt 6pt 0; }} figure.float .ph {{ width:100%!important; height:1.9in!important; }}
.ph {{ margin:0 auto; border:1.3pt dashed #8fa9c2; border-radius:4pt; background:repeating-linear-gradient(135deg,#f1f5f9,#f1f5f9 7pt,#e6edf4 7pt,#e6edf4 14pt); color:#5b7189; display:flex; align-items:center; justify-content:center; text-align:center; font:700 7.5pt/1.4 'Barlow Condensed'; letter-spacing:.8pt; }}
.ph b {{ font:600 8.5pt 'Public Sans'; letter-spacing:0; }}
.flowimg {{ max-width:100%; }}
.cover {{ page:cover; height:11in; position:relative; }} .cover .photo {{ height:5.9in; }} .cover .photo .ph {{ width:100%!important; height:100%!important; border-radius:0; border:none; }}
.cover .title {{ padding:.35in .7in 0; }} .cover h1.big {{ font-size:46pt; border:none; line-height:.98; margin:0 0 8pt; }}
.cover .sub {{ font:700 17pt 'Barlow Condensed'; color:var(--blue); }} .cover .lede {{ font-size:12pt; color:#33485d; margin-top:4pt; max-width:5.2in; }} .cover .ver {{ margin-top:10pt; font:600 12pt 'Barlow Condensed'; }}
.cover .photo img {{ width:100%!important; height:5.9in!important; max-height:none!important; max-width:none!important; object-fit:cover; }} .cover .logos img {{ max-height:.55in!important; }}
.cover .logos {{ position:absolute; left:.7in; right:.7in; bottom:.45in; display:flex; justify-content:space-between; }}
.credits {{ page:credits; break-before:page; padding-top:1.2in; max-width:5.6in; font-size:10pt; }}
.divider {{ page:divider; break-before:page; color:#fff; padding:3.4in .9in 0; height:6in; }} .divider .num {{ font:700 120pt/1 'Barlow Condensed'; color:var(--yellow); }} .divider h1 {{ color:#fff; border-color:var(--yellow); font-size:44pt; }}
.toc {{ break-before:page; }} .toc .row {{ display:block; margin:0; line-height:1.32; font-size:8.8pt; }} .toc a {{ color:inherit; text-decoration:none; }}
.toc a::after {{ content:leader('.') target-counter(attr(href), page); color:var(--blue); font-weight:700; }}
.toc .l0 {{ font:700 11.5pt 'Barlow Condensed'; color:var(--blue); margin-top:7pt; text-transform:uppercase; letter-spacing:.3pt; }} .toc .l1 {{ padding-left:14pt; font-weight:600; }} .toc .l2 {{ padding-left:30pt; font-size:8pt; }}
a.xref {{ color:var(--blue); text-decoration:none; font-weight:600; }} a.xref::after {{ content:', p. ' target-counter(attr(href), page); font-weight:400; }}
a.xref.t::after {{ content:' · p. ' target-counter(attr(href), page); }}
.topic {{ margin-top:30pt; }} .topic.np {{ break-before:page; margin-top:0; }}
.task {{ break-before:page; }} .task .head {{ display:flex; justify-content:space-between; align-items:flex-start; }} .task .code {{ font:600 11pt 'Barlow Condensed'; color:#6b7c8f; letter-spacing:.5pt; }}
.chips {{ margin:2pt 0 2pt; }} .chip {{ display:inline-block; font:600 8pt 'Barlow Condensed'; letter-spacing:.6pt; text-transform:uppercase; background:var(--yellow); color:#3a2a00; padding:1pt 7pt; border-radius:8pt; margin-right:3pt; white-space:nowrap; }}
p.when {{ font-size:8pt; color:#3b5068; margin:0 0 5pt; }}
.chip.type {{ background:#fff; border:1pt solid var(--blue); color:var(--blue); }} .chip.l0 {{ background:#dcebf8; color:#0a3d6b; }} .chip.l1 {{ background:#fff0c2; color:#5a4300; }} .chip.l2 {{ background:#fbd9c4; color:#7a2f0a; }}
.cols {{ display:table; table-layout:fixed; width:100%; border-spacing:0; }} .cols .main {{ display:table-cell; width:63%; vertical-align:top; padding-right:12pt; }} .cols .side {{ display:table-cell; width:37%; vertical-align:top; }}
.cols.ins .main {{ width:56%; }} .cols.ins .side {{ width:44%; text-align:center; }}
.cols2 {{ display:table; table-layout:fixed; width:100%; border-spacing:0; }} .cols2 .l {{ display:table-cell; width:41%; vertical-align:top; padding-right:12pt; text-align:center; }} .cols2 .r {{ display:table-cell; width:59%; vertical-align:top; }}
.box {{ border:1pt solid #c4d3e2; border-radius:5pt; margin-bottom:4pt; page-break-inside:avoid; text-align:left; }} .box.steps-box {{ page-break-inside:auto; }}
.box > .bh {{ background:var(--blue); color:#fff; font:700 10.5pt 'Barlow Condensed'; letter-spacing:.5pt; text-transform:uppercase; padding:2pt 8pt; }}
.box > .bb {{ padding:2pt 7pt 0pt; font-size:7.7pt; line-height:1.28; }} .box.warn > .bh {{ background:#B4361F; }} .box.warn {{ border-color:#e7b9b0; }}
.steps {{ list-style:none; margin:0; padding:0; counter-reset:s; }} .steps li {{ position:relative; padding-left:19pt; margin-bottom:3.2pt; font-size:8.4pt; line-height:1.3; }}
.steps li.n::before {{ counter-increment:s; content:counter(s); position:absolute; left:0; top:0; width:13pt; height:13pt; border-radius:50%; background:var(--blue); color:#fff; font:700 8.5pt/13pt 'Barlow Condensed'; text-align:center; }}
.steps li.q::before {{ background:var(--yellow); color:#3a2a00; }} .steps li.b {{ margin-left:14pt; padding-left:0; }} .steps li.sub {{ margin-left:14pt; }} .steps li.sub::before {{ content:'–'; position:absolute; left:-9pt; color:var(--blue); font-weight:700; }}
.tag {{ display:inline-block; font:700 7.5pt 'Barlow Condensed'; letter-spacing:.6pt; color:#fff; padding:0 5pt; border-radius:3pt; margin-right:4pt; }} .tag.y {{ background:#2f8f4e; }} .tag.nn {{ background:#b4361f; }}
ul.notes {{ margin:2pt 0 3pt; padding-left:12pt; font-size:8pt; color:#3b5068; }} .bb ul {{ margin:2pt 0 3pt; padding-left:12pt; }} .bb p {{ margin-bottom:3pt; }}
.basis {{ font-size:7.2pt; color:#566a7f; margin:3pt 0 0; text-align:left; }} .basis b {{ color:var(--blue); }}
.draft {{ font:600 7.5pt 'Barlow Condensed'; letter-spacing:.8pt; color:#B4361F; text-transform:uppercase; border:1pt solid #B4361F; border-radius:3pt; padding:0 5pt; }}
table.go td {{ font-size:8pt; font-weight:400; color:var(--ink); }} table.go td:first-child {{ font-weight:700; color:var(--blue); width:15%; }} table.go td.t a {{ display:block; }}
table.strip {{ width:100%; border-spacing:6pt 0; margin:4pt -6pt 0; table-layout:fixed; }} table.strip td {{ vertical-align:top; padding:0; border:none; background:#fff!important; font-weight:400; width:33%; }}
table.terms td {{ border:none; padding:2pt 6pt 2pt 0; background:#fff!important; font-weight:400; color:var(--ink); }} table.terms td:first-child {{ width:30%; }} table.terms tr {{ page-break-inside:avoid; }}
dl.gl {{ columns:3; column-gap:16pt; margin:4pt 0; }} dl.gl dt {{ font-weight:700; color:var(--blue); break-after:avoid; font-size:8.4pt; }} dl.gl dd {{ margin:0 0 4pt; font-size:7.9pt; line-height:1.32; break-inside:avoid; }}
.refs {{ columns:2; column-gap:18pt; font-size:7.9pt; }} .refs p {{ padding-left:12pt; text-indent:-12pt; word-break:break-word; margin-bottom:4pt; break-inside:avoid; }}
.appx {{ break-before:page; }} .appx .ph {{ width:100%!important; height:6.8in!important; }}
.topic h2.sec {{ margin-top:22pt; }} .topic h2.sec.group {{ margin-top:20pt; }} .topic p {{ margin-bottom:7.5pt; }} .topic .intro {{ margin-bottom:14pt; }} .topic table {{ margin-top:6pt; margin-bottom:12pt; }}
.big-fig {{ text-align:center; margin:2pt 0 8pt; }} .big-fig img {{ max-height:3.0in; max-width:100%; }}
table.ft {{ font-size:8pt; margin:0; }} table.ft td {{ padding:3pt 5pt; border:.6pt solid #c4d3e2; background:#fff!important; font-weight:400; color:var(--ink); }} table.ft td:first-child {{ width:auto; font-weight:700; color:var(--blue); }}
table.ft tr.sec td {{ background:var(--yellow)!important; color:#3a2a00; font:700 9pt 'Barlow Condensed'; letter-spacing:.5pt; text-transform:uppercase; padding:2pt 6pt; }} table.ft tr.tot td {{ background:var(--blue-l)!important; font-weight:700; color:var(--blue); }}
table.ft tr.blank td {{ height:.34in; }} table.ft td.ev {{ text-align:center; font-style:italic; color:#566a7f; background:#f4f8fb!important; }} table.ft tr.vis td {{ background:#fff8dc!important; font-weight:700; color:var(--blue); height:.3in; }} table.ft td.x {{ text-align:center; width:3.2%; padding:3pt 0; color:var(--blue); font-weight:700; }} table.ft th {{ padding:3pt 4pt; text-align:center; }}
.cl {{ page:landscape; break-before:page; }} .cl h1 {{ font-size:20pt; margin-bottom:5pt; }}
.cb {{ display:inline-block; width:8.5pt; height:8.5pt; border:1.1pt solid var(--blue); border-radius:1.5pt; vertical-align:-1.5pt; }}
table.hdr {{ margin:0 0 5pt; font-size:8.4pt; }} table.hdr td {{ border:.8pt solid #c4d3e2; padding:2pt 5pt; font-weight:400; color:var(--ink); width:auto; background:#fff!important; }} table.hdr td.k {{ background:var(--blue-l)!important; font-weight:700; color:var(--blue); width:11%; }}
table.clt {{ font-size:6.9pt; margin:0; }} table.clt th {{ font-size:7pt; padding:2pt 3pt; text-align:center; }} table.clt th.l {{ text-align:left; }} table.clt td {{ padding:1.6pt 3.5pt; border:.6pt solid #d3deea; }}
table.clt td:first-child {{ width:2.5%; text-align:center; color:var(--blue); background:#fff!important; }}
table.clt tr.sec td {{ background:var(--yellow)!important; color:#3a2a00; font:700 9pt 'Barlow Condensed'; letter-spacing:.5pt; text-transform:uppercase; text-align:left; padding:2pt 6pt; width:auto; }} table.clt tr.sec {{ page-break-after:avoid; }}
table.clt td.q {{ width:30%; font-weight:400; color:var(--ink); }} table.clt td.q b {{ color:var(--ink); }} .lk {{ color:#566a7f; margin-top:1.5pt; }}
table.clt td.c {{ text-align:center; width:3.6%; }} table.clt td.v {{ width:6.5%; white-space:nowrap; text-align:center; }} table.clt td.ab {{ width:27%; font-weight:400; color:var(--ink); }} table.clt td.tk {{ width:9%; font-weight:400; }} table.clt td.nt {{ width:10%; }}
.vb {{ display:inline-block; width:10pt; height:10pt; line-height:10pt; text-align:center; border-radius:50%; border:.8pt solid #b9c8d8; color:#b9c8d8; font:700 7pt 'Barlow Condensed'; margin:0 1pt; }} .vb.on {{ background:var(--blue); border-color:var(--blue); color:#fff; }}
.legend {{ font-size:7.6pt; color:#566a7f; margin-top:5pt; }} .foot {{ margin-top:5pt; padding:4pt 8pt; border:1pt solid #c4d3e2; border-radius:4pt; font-size:8pt; }} .foot b {{ color:var(--blue); }}
"""
print("CSS listo:", len(CSS), "caracteres")

In [ ]:
# --- Plantillas (una función por tipo de página) -------------------------------------------------------------------------------------
FREQ_ORDER = ["Routine", "Seasonal", "Biannual", "Annual", "Multiyear / Comprehensive", "As needed"]
VISIT_ORDER = ["Routine visit", "Seasonal visit", "Annual visit", "As needed (programmed)", "As needed (corrective)"]
def tpl_cover(s):
    return f'''<section class="cover"><div class="photo">{placeholder("COVER-PHOTO","100%","100%","COVER PHOTO")}</div>
    <div class="title"><h1 class="big">{esc(s["title"]).replace("MAINTENANCE","<br>MAINTENANCE")}</h1><div class="sub">{esc(s["subtitle"])}</div>
    <div class="lede">{esc(s["body"])}</div><div class="ver">{esc(VERSION)}</div></div>
    <div class="logos">{placeholder("LOGO-CMSF","1.9in","0.6in","LOGO")}{placeholder("LOGO-UCSC","1.9in","0.6in","LOGO")}</div></section>'''
def tpl_credits(s):
    return f'<section class="credits"><p class="src">{esc(s.get("version",""))}</p>' + "".join(f"<p>{esc(p)}</p>" for p in s["body"].split("\n\n")) + "</section>"
def tpl_divider(s, anchor): return f'<section class="divider" id="{anchor}"><div class="num">{esc(s.get("num",""))}</div><h1>{esc(s["title"])}</h1></section>'

def auto_table(system):
    groups = {}
    for t in D["tasks"]:
        if t["system"] == system: groups.setdefault(t["primary"], []).append(t)
    rows_ = ""
    for f in FREQ_ORDER:
        if f in groups:
            cells = "<br>".join(f'<a class="xref" href="#{t["id"]}">{esc(t["name"])}</a>' + (" (also after storms and at seasonal visits)" if t["kind"] == "Routine inspection" else "") for t in sorted(groups[f], key=lambda t: t["number"]))
            rows_ += f"<tr><td>{esc(f)}</td><td>{cells}</td></tr>"
    return f'<div class="keep"><table><thead><tr><th>Frequency</th><th>Tasks</th></tr></thead><tbody>{rows_}</tbody></table></div>'

def auto_visits():
    groups = {v: {} for v in VISIT_ORDER}
    for t in D["tasks"]:
        if t["visit"] in groups: groups[t["visit"]].setdefault(t["name"], []).append(t["system"])
    rows_ = ""
    for v in VISIT_ORDER:
        if groups[v]:
            cells = "<br>".join(f'{esc(n)} <span class="src">({", ".join("ch. 6" if x == "COR" else x for x in sorted(set(ss), key=lambda x: ["SB","TW","VD","WB","COR"].index(x)))})</span>' for n, ss in groups[v].items())
            rows_ += f"<tr><td>{esc(v)}</td><td>{cells}</td></tr>"
    return f'<table><thead><tr><th>Visit</th><th>Tasks</th></tr></thead><tbody>{rows_}</tbody></table>'

def render_section(sec):
    h, tx, typ = sec["heading"], sec["text"], sec["type"]; out = []
    is_cap = bool(h) and (h.endswith("by frequency") or "Guide" in h)
    if h and not tx and not sec["table"] and not sec["images"] and not sec["source"]: return ("group", f'<h2 class="sec group">{esc(h)}</h2>')
    if typ == "Image":
        return ("image", "".join(figure(i, tx, "100%", "1.9in", sec["layout"] or "float") if sec["layout"] != "full" else figure(i, tx, "6in", "3.3in") for i in (sec["images"] or [""])))
    if h:
        if is_cap: TAB["n"] += 1; h = f'Table {TAB["n"]}. {h}'
        out.append(f'<h2 class="sec{" cap" if is_cap else ""}">{esc(h)}</h2>')
    if sec["source"] == "auto:visits":
        return ("table", "".join(out) + (f"<p class='lead'>{md(tx, True)}</p>" if tx else "") + auto_visits())
    if sec["source"].startswith("auto:"):
        return ("table", "".join(out) + (f"<p class='lead'>{md(tx, True)}</p>" if tx else "") + auto_table(sec["source"][5:]))
    if typ == "Definition list":
        out.append(f"<p>{md(tx, True)}</p>" if tx else "")
        out.append('<dl class="gl">' + "".join(f"<dt>{esc(g['term'])}</dt><dd>{esc(g['definition'])}</dd>" for g in sorted(D["glossary"], key=lambda g: g["term"].lower())) + "</dl>")
        return ("text", "".join(out))
    if typ == "Terms":
        paras = [p.strip() for p in re.split(r"\n\n+", tx) if p.strip()]; rows_ = ""
        for p in paras:
            m = re.match(r"^(.{2,40}?) — (.+)$", p, re.S)
            if m: rows_ += f'<tr><td><span class="chip">{esc(m.group(1))}</span></td><td>{md(m.group(2), True)}</td></tr>'
            else: out.append(f"<p>{md(p, True)}</p>")
        out.append(f'<table class="terms"><tbody>{rows_}</tbody></table>'); return ("text", "".join(out))
    if typ == "Table":
        table = sec["table"]
        if is_cap: table = re.sub(r"(\|[^|\n]+\|)([^|\n]+)\|$", lambda m: m.group(1) + m.group(2).replace("; ", "@@BR@@") + "|", table, flags=re.M)
        t_html = xref(markdown.markdown(table, extensions=["tables"]).replace("@@BR@@", "<br>"))
        if len(table.strip().splitlines()) - 2 <= 4: t_html = f'<div class="keep">{t_html}</div>'
        if tx and tx.startswith("("): out += [t_html, f'<p class="src">{md(tx, True)}</p>']
        else: out += [f"<p class='lead'>{md(tx, True)}</p>" if tx else "", t_html]
        return ("table", "".join(out))
    if typ == "Numbered list":
        tx = re.sub(r"(?m)^(\d+)\.(?=\S)", r"\1. ", tx); m = re.search(r"\n\n(1\. The PDCA.*)$", tx, re.S); foot = ""
        if m: tx, foot = tx[:m.start()], m.group(1)[3:]
        out.append(md(tx))
        if foot: out.append(f'<p class="fn">1. {esc(foot)}</p>')
        return ("text", "".join(out))
    for p in re.split(r"\n\n+", tx):
        if p.strip(): out.append(f"<p>{md(p.strip(), True)}</p>")
    return ("text", "".join(out))

def tpl_topic(tid):
    t = TOPIC[tid]; sysb = (f'<span class="eyebrow" style="background:{SYS_COLOR[t["system"]]}">{esc(SYS_NAME[t["system"]])}</span><br>' if t["system"] in SYS_NAME else "")
    intro = f'<div class="intro">{md(t["intro"])}</div>' if t["intro"] else ""; blocks = []
    for s in SECS.get(tid, []):
        kind, h = render_section(s)
        if kind == "image" and s["layout"] != "full" and blocks and blocks[-1][0] == "text": blocks.insert(len(blocks) - 1, (kind, h))   # la figura flotante va antes del texto que acompaña
        else: blocks.append((kind, h))
    return f'<section class="topic{" np" if t["newpage"] else ""}" id="{tid}">{sysb}<h1>{esc(t["title"])}</h1>{intro}{"".join(b for _, b in blocks)}</section>'

def render_steps(tid):
    out = []
    for s in STEPS[tid]:
        tx, typ = s["text"], s["type"]
        if typ in ("Yes branch", "No branch"):
            tag = '<span class="tag y">YES</span>' if typ == "Yes branch" else '<span class="tag nn">NO</span>'
            out.append(f'<li class="b">{tag}{md(re.sub(r"^(yes|no)\s*[:\-\u2013\u2014]\s*", "", tx, flags=re.I), True)}</li>')
        elif s["level"] >= 2: out.append(f'<li class="sub">{md(tx, True)}</li>')
        else: out.append(f'<li class="n{" q" if typ.startswith("Decision") else ""}">{md(tx, True)}</li>')
    return '<ol class="steps" style="list-style:none">' + "".join(out) + "</ol>"
def box(title, content, cls=""): return f'<div class="box {cls}"><div class="bh">{esc(title)}</div><div class="bb">{content}</div></div>' if content else ""
def chips_html(t):
    lv = {"Observe / measure only": "l0", "Observe + minor action": "l1", "Action": "l2"}.get(t["level"], "")
    c = "".join(f'<span class="chip">{esc(f)}</span>' for f in t["freq"])
    if t["type"]: c += f'<span class="chip type">{esc(t["type"])}</span><span class="chip {lv}">{esc(t["level"])}</span>'
    return c
def head_html(t):
    draft = '<span class="draft">Draft · pending validation</span>' if t["eng"] != "Validated" else ""
    return (f'<div class="head"><div><span class="eyebrow" style="background:{SYS_COLOR[t["system"]]}">{esc(SYS_NAME[t["system"]])}</span></div><div>{draft} <span class="code">{t["id"]}</span></div></div>'
            f'<h1>{esc(t["name"])}</h1><div class="chips">{chips_html(t)}</div><p class="when">{md(t["fnotes"], True)}</p><div class="intro">{md(t["short"], True)}</div>')
def side_boxes(t): return box("Tools and equipment", field_html(t["tools"])) + box("Safety", field_html(t["safety"])) + box("Escalation", field_html(t["esc"]), "warn")

def basis_html(t):
    sc = {r["key"]: r["short"] for r in D["references"]}
    items_ = [sc.get(b) or b for b in t["basis"]]
    return f'<p class="basis"><b>Basis:</b> {esc("; ".join(items_))}.</p>' if items_ else ""

def tpl_task_plain(t):
    tid = t["id"]; img = t["images"][0] if t["images"] else f"FIG-{tid}"; fig = figure(img, "", "100%", "1.9in")
    if tid == "VD_T06":
        FIG["n"] += 1; fig = ""; big = f'<figure class="big-fig">{placeholder("FIG-VD_T06", "100%", "3.0in")}<figcaption><b>Figure {FIG["n"]}.</b> Ditch cross-section (schematic, not to scale).</figcaption></figure>'
        below = box("Responsibilities", field_html(t["resp"])) + box("Waste handling", field_html(t["waste"])) + box("Notes", field_html(t["notes"]))
        return (f'<section class="task" id="{tid}">{head_html(t)}{big}<div class="cols"><div class="main"><div class="box steps-box"><div class="bh">Steps</div><div class="bb">{render_steps(tid)}</div></div>{below}</div>'
                f'<div class="side">{side_boxes(t)}</div></div>{basis_html(t)}</section>')
    below = box("Responsibilities", field_html(t["resp"])) + box("Waste handling", field_html(t["waste"])) + box("Notes", field_html(t["notes"]))
    return (f'<section class="task" id="{tid}">{head_html(t)}<div class="cols"><div class="main"><div class="box steps-box"><div class="bh">Steps</div><div class="bb">{render_steps(tid)}</div></div>{below}</div>'
            f'<div class="side">{fig}{side_boxes(t)}</div></div>{basis_html(t)}</section>')
def tpl_task_flow(t):
    tid = t["id"]
    left = flow_figure(tid, "Steps to " + t["name"].lower() + ".", "4.7in") + box("Escalation", field_html(t["esc"]), "warn") + basis_html(t)
    right = box("Responsibilities", field_html(t["resp"])) + box("Tools and equipment", field_html(t["tools"])) + box("Safety", field_html(t["safety"])) + box("Waste handling", field_html(t["waste"])) + box("Notes", field_html(t["notes"]))
    return (f'<section class="task" id="{tid}">{head_html(t)}<div class="cols2"><div class="l">{left}</div>'
            f'<div class="r"><div class="box steps-box"><div class="bh">Steps</div><div class="bb">{render_steps(tid)}</div></div>{right}</div></div></section>')

def task_links(codes):
    return "".join(f'<div class="t"><a class="xref" href="#{c}">{esc(TASK[c]["name"])}</a></div>' for c in codes if c in TASK) or "—"
def tpl_inspection(t):
    tid, s = t["id"], t["system"]
    rows_ = "".join(f'<tr><td>{esc(r["section"])}</td><td>{esc(r["check"])}</td><td>{esc(r["abnormal"])}</td><td class="t">{task_links(r["go"])}</td></tr>' for r in D["look_at"].get(s, []))
    flow = flow_figure(tid, "What to do during a visual inspection.", "4.6in", "2.9in") if tid in FLOW_SVG else figure("FLOW-" + tid, "What to do during a visual inspection.", "100%", "3.2in")
    strip = f'<table class="strip"><tr><td>{box("Tools and equipment", field_html(t["tools"]))}</td><td>{box("Safety", field_html(t["safety"]))}</td><td>{box("Escalation", field_html(t["esc"]), "warn")}</td></tr></table>'
    return (f'<section class="task" id="{tid}">{head_html(t)}<div class="cols ins"><div class="main"><div class="box steps-box"><div class="bh">Steps</div><div class="bb">{render_steps(tid)}</div></div></div><div class="side">{flow}</div></div>'
            f'<div class="keep"><h2 class="sec cap">What to look at and what to do</h2><table class="go"><thead><tr><th>Look at</th><th>What to check</th><th>If abnormal</th><th>Go to task</th></tr></thead><tbody>{rows_}</tbody></table></div>{strip}{basis_html(t)}</section>')
def tpl_task(tid):
    t = TASK[tid]
    if t["kind"] == "Routine inspection": return tpl_inspection(t)
    return tpl_task_flow(t) if tid in FLOW_SVG else tpl_task_plain(t)

def tpl_refs():
    items = sorted(D["references"], key=lambda r: re.sub(r"[^a-z]", "", r["apa"].lower()))
    return '<section class="topic np" id="REFS"><h1>References</h1><div class="refs">' + "".join(f"<p>{esc(r['apa'])}</p>" for r in items) + "</div></section>"
def form_head(code, letter, title):
    return f'<span class="eyebrow" style="background:{BLUE}">Appendix {letter} · Form {code}</span><h1>{esc(title)}</h1>'
def trim(t, n):
    t = (t or "").strip()
    return t if len(t) <= n else t[:n].rsplit(" ", 1)[0].rstrip(",;:") + "…"
def months_of(t): return forms_xlsx.parse_months(t["months"])
def tpl_appendix(code, letter):
    sysc = code[:2]; kind = code[3:]
    ftr = f'<p class="legend"><span class="draft">Draft</span> Editable Excel version: VTS_forms.xlsx (sheet {code}).</p>'
    if code == "VTS_F01":
        rows_ = "".join(f'<tr><td>{esc(d["code"])}</td><td>{esc(d["type"])}</td><td>{esc(d["name"])}</td><td>{esc(d["desc"])}</td><td>{esc(d["version"])}</td><td>{esc(d["status"])}</td></tr>' for d in D["doc_rows"])
        return f'<section class="cl" id="{code}">{form_head(code, letter, "Document Control Sheet")}<table class="ft"><thead><tr><th>Document</th><th>Type</th><th>Title</th><th>Description</th><th>Version</th><th>Status</th></tr></thead><tbody>{rows_}</tbody></table>{ftr}</section>'
    if kind == "F02" and sysc in SYS_NAME:
        ts = sorted([t for t in D["tasks"] if t["system"] == sysc], key=lambda t: t["number"]); rows_ = ""
        for f in FREQ_ORDER:
            g = [t for t in ts if t["primary"] == f]
            if g:
                rows_ += f'<tr class="sec"><td colspan="5">{esc(f)}</td></tr>' + "".join(f'<tr><td>{t["id"]}</td><td>{esc(t["name"])}</td><td>{esc(t["component"])}</td><td>{esc(t["visit"])}</td><td>{esc(t["f02"])}</td></tr>' for t in g)
        hdr = f'<table class="hdr"><tr><td class="k">VTS type</td><td>{esc(SYS_NAME[sysc])}</td><td class="k">Prepared by</td><td></td><td class="k">Date</td><td></td></tr></table>'
        return f'<section class="cl" id="{code}">{form_head(code, letter, f"Preventive Maintenance Reference — {SYS_NAME[sysc]}")}{hdr}<table class="ft"><thead><tr><th>Task</th><th>Task name</th><th>Component</th><th>Visit</th><th>What to check / how to do it</th></tr></thead><tbody>{rows_}</tbody></table>{ftr}</section>'
    if kind == "F03" and sysc in SYS_NAME:
        ts = sorted([t for t in D["tasks"] if t["system"] == sysc], key=lambda t: t["number"]); counts = [0] * 12
        rows_ = '<tr class="vis"><td colspan="3">Planned visits (write the dates)</td>' + "".join('<td class="x"></td>' for _ in range(12)) + "<td>Routine tasks are done at every visit.</td></tr>"
        for t in ts:
            ms = months_of(t); lead = f'<td>{esc(t["primary"])}</td><td>{t["id"]} {esc(t["name"])}</td><td>{esc(t["component"])}</td>'
            if t["primary"] == "Routine": mid = '<td colspan="12" class="ev">at every scheduled visit (and after significant storms for the inspection)</td>'
            elif not ms: mid = '<td colspan="12" class="ev">as needed</td>'
            else:
                for i in ms: counts[i] += 1
                mid = "".join(f'<td class="x">{"x" if i in ms else ""}</td>' for i in range(12))
            rows_ += f'<tr>{lead}{mid}<td>{esc(trim(t["fnotes"], 95))}</td></tr>'
        rows_ += '<tr class="tot"><td colspan="3">Seasonal and annual tasks scheduled this month (count)</td>' + "".join(f'<td class="x">{c}</td>' for c in counts) + "<td></td></tr>"
        hdr = f'<table class="hdr"><tr><td class="k">VTS type</td><td>{esc(SYS_NAME[sysc])}</td><td class="k">Site</td><td></td><td class="k">Prepared by / date</td><td></td></tr></table>'
        head = "<thead><tr><th>Frequency</th><th>Task</th><th>Component</th>" + "".join(f"<th>{m}</th>" for m in forms_xlsx.MONTHS) + "<th>Notes</th></tr></thead>"
        return f'<section class="cl" id="{code}">{form_head(code, letter, f"Annual Plan — {SYS_NAME[sysc]}")}{hdr}<table class="ft">{head}<tbody>{rows_}</tbody></table>{ftr}</section>'
    if code in ("VTS_F05", "VTS_F06"):
        cols = [(c, w) for f, o, c, w in sorted(D["forms"], key=lambda x: (x[0], x[1])) if f == code]; tot = sum(w for _, w in cols)
        ttl = "Corrective Maintenance Log" if code == "VTS_F05" else "Maintenance Log"
        hdr = '<table class="hdr"><tr><td class="k">VTS name</td><td></td><td class="k">VTS type</td><td></td><td class="k">Supervisor</td><td></td></tr></table>'
        head = "<thead><tr>" + "".join(f'<th style="width:{w / tot * 100:.1f}%">{esc(c)}</th>' for c, w in cols) + "</tr></thead>"
        body = "".join('<tr class="blank">' + "<td></td>" * len(cols) + "</tr>" for _ in range(11))
        return f'<section class="cl" id="{code}">{form_head(code, letter, ttl)}{hdr}<table class="ft">{head}<tbody>{body}</tbody></table>{ftr}</section>'
    return f'<section class="appx" id="{code}"><span class="eyebrow" style="background:{BLUE}">Appendix {letter}</span><h1>{esc(code)} {esc(DOC[code])}</h1>{placeholder("FORM-"+code, "100%", "6.8in", "FORM")}</section>'

def tpl_checklist(system, letter):
    items = CHECK.get(system, []); cb = '<span class="cb"></span>'
    hdr = (f'<table class="hdr"><tr><td class="k">Site / VTS ID</td><td></td><td class="k">Date</td><td></td><td class="k">Inspector</td><td></td></tr>'
           f'<tr><td class="k">Visit type</td><td colspan="3">{cb} Routine &nbsp; {cb} After storm &nbsp; {cb} Seasonal</td><td class="k">Last rain (date / amount)</td><td></td></tr></table>')
    sp = D["site_params"].get(system, [])
    if sp:
        trs = ""
        for k in range(0, len(sp), 3):
            chunk = sp[k:k + 3]; cells = "".join(f'<td class="k">{esc(f)}</td><td></td>' for f in chunk) + "".join('<td class="k"></td><td></td>' for _ in range(3 - len(chunk)))
            trs += f"<tr>{cells}</tr>"
        hdr += f'<table class="hdr"><tr><td class="k" colspan="6" style="width:100%;text-align:left">Site conditions (the supervisor fills these in from the design or the annual plan)</td></tr>{trs}</table>'
    rows_, cur, n = [], None, 0
    for it in items:
        if it["section"] != cur: cur = it["section"]; rows_.append(f'<tr class="sec"><td colspan="9">{esc(cur)}</td></tr>')
        n += 1
        vis = "".join(f'<span class="vb{" on" if v in it["visits"] else ""}">{c}</span>' for c, v in (("R", "Routine"), ("A", "After storm"), ("S", "Seasonal")))
        tasks = "<br>".join(f'<a class="xref t" href="#{c}">{c}</a>' for c in it["go"] if c in TASK) or ""
        normal = f'<div class="lk"><b>Normal:</b> {esc(it["normal"])}</div>' if it["normal"] else ""
        rows_.append(f'<tr><td>{n}</td><td class="q"><b>{esc(it["question"])}</b><div class="lk">{esc(it["look"])}</div>{normal}</td><td class="c">{cb}</td><td class="c">{cb}</td><td class="c">{cb}</td><td class="v">{vis}</td>'
                     f'<td class="ab">{esc(it["abnormal"])}</td><td class="tk">{tasks}</td><td class="c">{cb if it["photo"] else ""}</td><td class="nt"></td></tr>')
    head = '<thead><tr><th>#</th><th class="l">Item</th><th>OK</th><th>Abn.</th><th>N/A</th><th>Visit</th><th class="l">If abnormal</th><th class="l">Go to task</th><th>Photo</th><th class="l">Notes</th></tr></thead>'
    foot = f'<div class="foot"><b>Before you leave:</b> &nbsp; {cb} Every abnormal finding is recorded and photographed &nbsp; {cb} Supervisor notified &nbsp; {cb} Follow-up task(s) chosen: ______________________ &nbsp; Inspector signature: ______________</div>'
    return (f'<section class="cl" id="{system}_F04"><span class="eyebrow" style="background:{SYS_COLOR[system]}">Appendix {letter} · Form {system}_F04</span><h1>Visual Inspection Checklist — {esc(SYS_NAME[system])}</h1>'
            f'{hdr}<table class="clt">{head}<tbody>{"".join(rows_)}</tbody></table>{foot}'
            f'<p class="legend">Visit: <b>R</b> = Routine · <b>A</b> = After storm · <b>S</b> = Seasonal. Observe and document; do only the small actions listed. <span class="draft">Draft</span> What "normal" looks like and site-specific values are still to be confirmed.</p></section>')

In [ ]:
# --- Armado del libro ------------------------------------------------------------------------------------------------------
def build_toc(order):
    rows_, ch, sub, tsub, appx = [], None, 0, 0, 0
    L = lambda lvl, label, title, anchor: rows_.append(f'<div class="row l{lvl}"><a href="#{anchor}">{esc(label + "  " if label else "")}{esc(title)}</a></div>')
    for typ, ref in order:
        if typ == "Separator":
            s = D["separators"][ref]
            if ref.startswith("Divider - Appendices"): L(0, "", "Appendices", "DIV-APPX"); ch = "A"
            elif ref.startswith("Divider"): ch = s["num"]; sub = 0; L(0, ch + ".", s["title"], "DIV-" + ch)
        elif typ == "Topic":
            if ref == "TOP-01": L(0, "0.", TOPIC[ref]["title"], ref); continue
            if ref == "TOP-02": ch, sub = "1", 0
            if ref == "TOP-16": L(0, "6.", TOPIC[ref]["title"], ref); ch = "6"; sub = 0; tsub = 0; continue
            sub += 1; tsub = 0
            if ch == "1" and sub == 1: rows_.append(f'<div class="row l0"><a href="#{ref}">1.  Introduction</a></div>')
            L(1, f"{ch}.{sub}", TOPIC[ref]["title"], ref)
        elif typ == "Task":
            tsub += 1
            if ch == "6": L(1, f"6.{tsub}", f'{ref}  {TASK[ref]["name"]}', ref)
            else: L(2, f"{ch}.{sub}.{tsub}", f'{ref}  {TASK[ref]["name"]}', ref)
        elif typ == "References": L(0, "7.", "References", "REFS")
        elif typ == "Document": letter = chr(ord("A") + appx); appx += 1; L(1, f"Appendix {letter}", f"{ref} {DOC[ref]}", ref)
        elif typ == "Checklist": letter = chr(ord("A") + appx); appx += 1; L(1, f"Appendix {letter}", f"{ref}_F04 Visual Inspection Checklist — {SYS_NAME[ref]}", f"{ref}_F04")
    return '<section class="toc"><h1>Table of Contents</h1>' + "".join(rows_) + "</section>"

def build_html():
    FIG["n"] = 0; TAB["n"] = 0
    order = [(t, r) for t, r in D["manual_org"] if DRAFT_MODE or t != "Task" or TASK[r]["eng"] == "Validated"]
    parts, appx = [], 0
    for typ, ref in order:
        if typ == "Separator":
            s = D["separators"][ref]
            if ref == "Cover": parts.append(tpl_cover(s))
            elif ref == "Credits": parts.append(tpl_credits(s))
            elif ref == "Table of contents": parts.append(build_toc(order))
            elif ref.startswith("Divider - Appendices"): parts.append(tpl_divider(s, "DIV-APPX"))
            elif ref.startswith("Divider"): parts.append(tpl_divider(s, "DIV-" + s["num"]))
        elif typ == "Topic": parts.append(tpl_topic(ref))
        elif typ == "Task": parts.append(tpl_task(ref))
        elif typ == "References": parts.append(tpl_refs())
        elif typ == "Document": parts.append(tpl_appendix(ref, chr(ord("A") + appx))); appx += 1
        elif typ == "Checklist": parts.append(tpl_checklist(ref, chr(ord("A") + appx))); appx += 1
    return f'<!doctype html><html lang="en"><head><meta charset="utf-8"><title>VTS O&amp;M Manual</title><style>{CSS}</style></head><body>' + "".join(parts) + "</body></html>"
from schematics import make_vd_cross_section
make_vd_cross_section(IMAGES_DIR / "FIG-VD_T06.svg")
HTML = build_html(); (OUT_DIR / "book.html").write_text(HTML, encoding="utf-8"); print("HTML:", len(HTML) // 1024, "KB")

In [ ]:
# --- PDF ---------------------------------------------------------------------------------------------------------------------
from weasyprint import HTML as WP
doc = WP(string=HTML, base_url=str(BASE_DIR)).render(); doc.write_pdf(str(OUT_PDF))
pages_of = {}
for i, p in enumerate(doc.pages, 1):
    for a in getattr(p, "anchors", {}) or {}: pages_of.setdefault(a, i)
print(f"PDF: {len(doc.pages)} páginas → {OUT_PDF}")
anchors = []
for typ, ref in D["manual_org"]:
    if typ == "Separator" and ref.startswith("Divider"): anchors.append("DIV-APPX" if "Appendices" in ref else "DIV-" + D["separators"][ref]["num"])
    elif typ in ("Topic", "Task", "Document"): anchors.append(ref)
    elif typ == "Checklist": anchors.append(f"{ref}_F04")
    elif typ == "References": anchors.append("REFS")
long_ = [(a, pages_of[b] - pages_of[a]) for a, b in zip(anchors, anchors[1:]) if a in TASK and a in pages_of and b in pages_of and pages_of[b] - pages_of[a] > 1 and TASK[a]["kind"] != "Routine inspection"]
print("Tareas que ocupan más de 1 página:", long_ or "ninguna")

## Excel / Google Sheets
Un checklist por sistema con el mismo diseño del anexo: casillas ☐/☒, colores, menú de tipo de visita y resumen.
En Google Drive: *Archivo → Importar → Subir* (o abrir el .xlsx con Hojas de cálculo de Google).

In [ ]:
from xlsx_checklist import write_checklist_xlsx
TASK_NAMES = {t["id"]: t["name"] for t in D["tasks"]}
for sy in CHECK: print("Excel:", write_checklist_xlsx(sy, CHECK[sy], SYS_NAME[sy], SYS_COLOR[sy], TASK_NAMES, D["site_params"].get(sy, []), OUT_DIR / f"{sy}_F04_visual_inspection_checklist.xlsx"))

## Formularios en Excel (F01, F02, F03, F05, F06)
Un solo libro con una hoja por formulario. F03 trae fórmulas que cuentan las tareas de cada mes.

In [ ]:
tasks_f = [dict(id=t["id"], system=t["system"], number=t["number"], name=t["name"], primary=t["primary"], component=t["component"], visit=t["visit"], f02=t["f02"], months=t["months"], when=t["fnotes"]) for t in D["tasks"]]
print("Excel:", forms_xlsx.write_forms_xlsx(OUT_DIR / "VTS_forms.xlsx", D["doc_rows"], tasks_f, sorted(D["forms"], key=lambda x: (x[0], x[1])), SYS_NAME, FREQ_ORDER))